A CHATBOT THAT HELPS YOU FIND AFFORDABLE PROPERTIES IN DIFFERENT CITIES

In [1]:
from dotenv import load_dotenv
import os
from openai import OpenAI 
import gradio as gr
import json
import sqlite3 

In [18]:
# Initialization

load_dotenv(override=True)

api_key = os.getenv('GOOGLE_API_KEY')
if api_key:
    print(f"Google API Key exists and begins {api_key[:8]}")
else:
    print("Google API Key not set")
    
MODEL = "gemini-2.5-flash-lite" 
url='https://generativelanguage.googleapis.com/v1beta/openai/'

gemini = OpenAI(base_url=url, api_key=api_key)

DB = "real_estate.db"

Google API Key exists and begins AIzaSyA-


In [3]:
system_prompt = """ Your are a helpful Real Estate Bot, that stores location, price and 
size of the property in a database. The bot should be able to answer questions about the properties, 
such as "What is the average price of properties in New York?" or "How many properties are larger than 1000 sqft?" """

# real estate bot, that stores location, price and size of the property in a database. The bot should be able to answer questions about the 
# properties, such as "What is the average price of properties in New York?" or "How many properties are larger than 1000 sqft?"
# data base will have 3 columns: location, price and size
# tools used will be: for retreiving 3 properties
# gradio interface chat and reply

In [4]:

with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()
    cursor.execute('CREATE TABLE IF NOT EXISTS real_estate (location TEXT PRIMARY KEY, price REAL, size REAL)')
    conn.commit()

In [5]:
def get_estate_info(location):
    print(f"DATABASE TOOL CALLED: Getting price for {location}", flush=True) 
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('SELECT price, size FROM real_estate WHERE location = ?', (location.lower(),))
        result = cursor.fetchone()
        return f"price and size of the property at {location} is {result[0]} and ${result[1]} sq ft" if result else "No price data available for this location"

In [111]:
get_estate_info("New York")

DATABASE TOOL CALLED: Getting price for New York


'No price data available for this location'

In [6]:
def set_estate_info(location, price, size):
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('INSERT INTO real_estate (location, price, size) VALUES (?, ?, ?) ON CONFLICT(location) DO UPDATE SET price = ?, size = ?', (location.lower(), price, size, price, size))
        conn.commit()

In [108]:
#  to delete items in db
with sqlite3.connect(DB) as conn:
    conn.execute("DELETE FROM real_estate")
    conn.commit()

In [8]:
def check_db():
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        # Fetch every row from the real_estate table
        cursor.execute("SELECT * FROM real_estate")
        rows = cursor.fetchall()
        
        if not rows:
            print("The database is currently empty.")
        else:
            print(f"Found {len(rows)} records:")
            for row in rows:
                print(row)

check_db()

Found 4 records:
('sangli', 8000.0, 800.0)
('ohio', 20000.0, 1100.0)
('london', 100000.0, 650.0)
('tokyo', 80000.0, 700.0)


In [9]:
estate_info = [
    {"location": "sangli", "price": 8000, "size": 800},
    {"location": "ohio", "price": 20000, "size": 1100},
    {"location": "london", "price": 100000, "size": 650},
    {"location": "tokyo", "price": 80000, "size": 700}
]

In [115]:
# 1. this only works when there is single parameter, input through sets in  key value pair
# for location, price, size in estate_info.items():
#     set_estate_info(location, price, size)

# 2. this is when there are more than 1 parameters and input is using lists


for item in estate_info:
    # We pass the values from the dictionary keys to your function
    set_estate_info(item["location"], item["price"], item["size"])

In [10]:
def get_cheapest_property():
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('Select location, price FROM real_estate ORDER BY price ASC LIMIT 1')
        result = cursor.fetchone()
        return f"The cheapest property is located at {result[0]} with a price of {result[1]}" if result else "No properties found in the database"
        

In [20]:
def count_properties_by_size(min_size, max_size):

    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()

        if min_size and max_size:
            cursor.execute(
            "SELECT COUNT(*) FROM real_estate WHERE size BETWEEN ? AND ?",
            (min_size, max_size)
            )

        elif min_size:
            cursor.execute(
            "SELECT COUNT(*) FROM real_estate WHERE size >= ?",
            (min_size,)
            )

        elif max_size:
            cursor.execute(
            "SELECT COUNT(*) FROM real_estate WHERE size <= ?",
            (max_size,)
            )

        else:
            cursor.execute("SELECT COUNT(*) FROM real_estate")

        result = cursor.fetchone()

        return f"There are {result[0]} matching properties."

In [21]:
estate_function = {
    "name": "get_estate_info",
    "description": "Get the price and size of an estate in a given location.",
    "parameters": {
        "type": "object",
        "properties": {
            "location": {
                "type": "string",
                "description": "The place buyer is interested in buying the property",
            },
        },
        "required": ["location"],
        "additionalProperties": False
    }
}

cheap_function = {
    "name": "get_cheapest_property",
    "description": "Get the location and price of the cheapest property in the database.",
    "parameters": {
        "type": "object",
        "properties": {
            "price": {
                "type": "number",
                "description": "The price of the cheapest property in the database."
            },
            "location": {
                "type": "string",
                "description": "The location of the cheapest property in the database." 
            }
        },
        "required": ["price"],
        "additionalProperties": False
    }
}

count_size_function = {
    "name": "count_properties_by_size",
    "description": "Count properties by size conditions like above, below, or between sizes.",
    "parameters": {
        "type": "object",
        "properties": {
            "min_size": {
                "type": "number",
                "description": "Minimum property size in square feet"
            },
            "max_size": {
                "type": "number",
                "description": "Maximum property size in square feet"
            }
        },
        "required": ["min_size", "max_size"],
        "additionalProperties": False
    }
}
# , {"type": "function", "function": count_size_function}
tools = [{"type": "function", "function": estate_function}, {"type": "function", "function": cheap_function}, {"type": "function", "function": count_size_function}]
tools

[{'type': 'function',
  'function': {'name': 'get_estate_info',
   'description': 'Get the price and size of an estate in a given location.',
   'parameters': {'type': 'object',
    'properties': {'location': {'type': 'string',
      'description': 'The place buyer is interested in buying the property'}},
    'required': ['location'],
    'additionalProperties': False}}},
 {'type': 'function',
  'function': {'name': 'get_cheapest_property',
   'description': 'Get the location and price of the cheapest property in the database.',
   'parameters': {'type': 'object',
    'properties': {'price': {'type': 'number',
      'description': 'The price of the cheapest property in the database.'},
     'location': {'type': 'string',
      'description': 'The location of the cheapest property in the database.'}},
    'required': ['price'],
    'additionalProperties': False}}},
 {'type': 'function',
  'function': {'name': 'count_properties_by_size',
   'description': 'Count properties by size condit

In [22]:
def handle_tool_calls(message):
    responses = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_estate_info":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get('location')
            price_details = get_estate_info(city)
            responses.append({
                "role": "tool",
                "content": price_details,
                "tool_call_id": tool_call.id
            })
        elif tool_call.function.name == "get_cheapest_property":
            price_details = get_cheapest_property()
            responses.append({
                "role": "tool",
                "content": price_details,
                "tool_call_id": tool_call.id
            })
        elif tool_call.function.name == "count_properties_by_size":
            arguments = json.loads(tool_call.function.arguments)
            min_size = arguments.get('min_size')
            max_size = arguments.get('max_size')
            count_details = count_properties_by_size(min_size, max_size)
            responses.append({
                "role": "tool",
                "content": count_details,
                "tool_call_id": tool_call.id
            })
    return responses

In [23]:
def chat(message, history):
    try:
        history = [{"role":h["role"], "content":h["content"]} for h in history]

        messages = [{"role": "system", "content": system_prompt}] + history + [
            {"role": "user", "content": message}
        ]

        response = gemini.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=tools
        )

        while response.choices[0].finish_reason == "tool_calls":
            message = response.choices[0].message
            responses = handle_tool_calls(message)

            messages.append(message)
            messages.extend(responses)

            response = gemini.chat.completions.create(
                model=MODEL,
                messages=messages,
                tools=tools
            )

        return response.choices[0].message.content

    except Exception as e:
        print("ERROR:", e)
        return f"Error occurred: {str(e)}"

In [25]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.
